# **Setup**

## **Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report
)
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import shap
import lime
import lime.lime_tabular

## **Loading Data**

In [ ]:
base_path = '/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/'

circuits = pd.read_csv(base_path + 'circuits.csv')
constructor_results = pd.read_csv(base_path + 'constructor_results.csv')
constructor_standings = pd.read_csv(base_path + 'constructor_standings.csv')
constructors = pd.read_csv(base_path + 'constructors.csv')
driver_standings = pd.read_csv(base_path + 'driver_standings.csv')
drivers = pd.read_csv(base_path + 'drivers.csv')
lap_times = pd.read_csv(base_path + 'lap_times.csv')
pit_stops = pd.read_csv(base_path + 'pit_stops.csv')
qualifying = pd.read_csv(base_path + 'qualifying.csv')
races = pd.read_csv(base_path + 'races.csv')
results = pd.read_csv(base_path + 'results.csv')
seasons = pd.read_csv(base_path + 'seasons.csv')
sprint_results = pd.read_csv(base_path + 'sprint_results.csv')
status = pd.read_csv(base_path + 'status.csv')

In [ ]:
tables = {
    'circuits': circuits,
    'constructor_results': constructor_results,
    'constructor_standings': constructor_standings,
    'constructors': constructors,
    'driver_standings': driver_standings,
    'drivers': drivers,
    'lap_times': lap_times,
    'pit_stops': pit_stops,
    'qualifying': qualifying,
    'races': races,
    'results': results,
    'seasons': seasons,
    'sprint_results': sprint_results,
    'status': status
}

# **Data Cleaning & Auditing**

## **Initial EDA**

In [ ]:
summary = []
missing = []
sentinels = []
duplicates = []

for name, df in tables.items():
    summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    })
    
    missing.append({
        "dataset": name,
        "missing_values": df.isna().sum().sum()
    })
    
    sentinels.append({
        "dataset": name,
        "sentinel_values": (df == r"\N").sum().sum()
    })
    
    duplicates.append({
        "dataset": name,
        "duplicates": df.duplicated().sum()
    })


check = bool(tables['results'][['raceId', 'driverId']].duplicated().any())
number = tables['results'][['raceId', 'driverId']].duplicated().sum()
duplicate_groups = (
    tables['results'].groupby(["raceId", "driverId"])
    .size()
)
duplicate_groups = duplicate_groups[duplicate_groups > 1]

check_car_dupe = bool(tables['results'][['raceId', 'number']].duplicated().any())
number_car_dupe = tables['results'][['raceId', 'number']].duplicated().sum()
duplicate_cars_groups = (
    tables['results'].groupby(["raceId", "number"])
    .size()
)
duplicate_cars_groups = duplicate_cars_groups[duplicate_cars_groups > 1]

summary_df = pd.DataFrame(summary)
missing_df = pd.DataFrame(missing)
sentinels_df = pd.DataFrame(sentinels)
duplicates_df = pd.DataFrame(duplicates)

print('Does the results table contain any duplicate driver race data?: ', check)
print('How many? ', number)
print("Driver-race combinations with duplicates:", len(duplicate_groups))

print('\n')

print('Does the results table contain any duplicate car race data?: ', check_car_dupe)
print('How many? ', number_car_dupe)
print("Driver-car combinations with duplicates:", len(duplicate_cars_groups))

print('\n')

print('########### Missing Columns ####################')
for name, df in tables.items():
    missing_cols = df.isna().sum()
    missing_cols = missing_cols[missing_cols > 0]

    if len(missing_cols) > 0:
        print(f"\n{name}")
        print(missing_cols)

print('\n')

print('###################### Sentinel Columns ###################')
for name, df in tables.items():
    sentinel_cols = (df == r"\N").sum()
    sentinel_cols = sentinel_cols[sentinel_cols > 0]

    if len(sentinel_cols) > 0:
        print(f"\n{name}")
        print(sentinel_cols)

print('\n')

# Create a 2x2 grid of subplots with a shared figure size
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 12))

# 1. Dataset Dimensions (Top-Left)
summary_df.plot(
    x="dataset",
    y=["rows"],
    kind="bar",
    ax=axes[0, 0],
    title="Dataset Dimensions"
)

# 2. Missing Values (Top-Right)
missing_df.plot(
    x="dataset",
    y="missing_values",
    kind="bar",
    ax=axes[0, 1],
    title="Missing Values by Dataset",
    legend=False
)

# 3. Sentinel Values (Bottom-Left)
sentinels_df.plot(
    x="dataset",
    y="sentinel_values",
    kind="bar",
    ax=axes[1, 0],
    title="Sentinel Values by Dataset",
    legend=False
)

# 4. Duplicate Rows (Bottom-Right)
duplicates_df.plot(
    x="dataset",
    y="duplicates",
    kind="bar",
    ax=axes[1, 1],
    title="Duplicate Rows by Dataset",
    legend=False
)

# Rotate x-axis labels for readability and prevent overlapping titles/labels
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
for key, value in tables.items():
    print(f'###### {key} #######')
    print(value.info())
    print(value.describe())
    display(value.head(5))

## **String Normalization**

In [ ]:
for column in df.columns:
    if(df[column].dtype == object):
        df[column] = df[column].str.strip()
        df[column] = df[column].replace(r'^\s*$', np.nan, regex=True)

## **Numerical Columns Casting**

In [ ]:
cleaned_tables = {}

def convert_column_to_number(table):
    for column in table.columns:
        new_column = pd.to_numeric(table[column], errors='coerce')
        if new_column.notna().sum() == table[column].notna().sum():
            table[column] = new_column
    return table

# Convert Number Columns To Number Type After Removing \N 
for table_name,table in tables.items():
    cleaned_tables[table_name] = convert_column_to_number(table.replace("\\N", np.nan))

In [ ]:
for key, value in cleaned_tables.items():
    print(f'###### {key} #######')
    print(value.info())
    display(value.head(5))

## **Date Columns Casting**

In [ ]:
# Convert Date Strings To Date Objects
cleaned_tables['races']['date'] = pd.to_datetime(cleaned_tables['races']['date'])
cleaned_tables['drivers']['dob'] = pd.to_datetime(cleaned_tables['drivers']['dob'])

## **Lap Times to Seconds Normalization**

In [ ]:
# Convert Lap Times To Seconds
def time_to_seconds(text):
    if pd.isna(text):
        return np.nan
    minutes, seconds = text.split(':')
    return float(minutes) * 60 + float(seconds)


for col in ['q1', 'q2', 'q3']:
    cleaned_tables['qualifying'][col] = cleaned_tables['qualifying'][col].map(time_to_seconds)

In [ ]:
cleaned_tables['qualifying'].head()

## **Referential Integrity**

In [ ]:
# Create Composite Primary Keys
cleaned_tables['lap_times']['lapId'] = (cleaned_tables['lap_times']['raceId'].astype(str) + '_' + cleaned_tables['lap_times']['driverId'].astype(str) + '_' + cleaned_tables['lap_times']['lap'].astype(str))
cleaned_tables['pit_stops']['stopId'] = (cleaned_tables['pit_stops']['raceId'].astype(str) + '_' + cleaned_tables['pit_stops']['driverId'].astype(str) + '_' + cleaned_tables['pit_stops']['stop'].astype(str))

primary_keys = {
    'circuits': 'circuitId', 
    'constructor_results': 'constructorResultsId', 
    'constructor_standings': 'constructorStandingsId',
    'constructors': 'constructorId', 
    'driver_standings': 'driverStandingsId', 
    'drivers': 'driverId',
    'lap_times': 'lapId',
    'pit_stops': 'stopId',
    'qualifying': 'qualifyId', 
    'races': 'raceId',
    'results': 'resultId', 
    'seasons': 'year',
    'sprint_results': 'resultId',
    'status': 'statusId', 
}


for table_name, key in primary_keys.items():
    unique = cleaned_tables[table_name][key].is_unique
    no_missing = cleaned_tables[table_name][key].notna().all()
    print(f'{table_name}.{key}: \n unique = {unique} \n no missing = {no_missing}')

In [ ]:
foreign_keys = [
    
    ('constructor_results', 'raceId', 'races', 'raceId'),
    ('constructor_results', 'constructorId', 'constructors', 'constructorId'),

    ('constructor_standings', 'raceId', 'races', 'raceId'),
    ('constructor_standings', 'constructorId', 'constructors', 'constructorId'),

    ('driver_standings', 'raceId', 'races', 'raceId'),
    ('driver_standings', 'driverId', 'drivers', 'driverId'),

    ('lap_times', 'raceId', 'races', 'raceId'),
    ('lap_times', 'driverId', 'drivers', 'driverId'),

    
    ('pit_stops', 'raceId', 'races', 'raceId'),
    ('pit_stops', 'driverId', 'drivers', 'driverId'),

    ('qualifying', 'raceId', 'races', 'raceId'),
    ('qualifying', 'driverId', 'drivers', 'driverId'),
    ('qualifying', 'constructorId', 'constructors', 'constructorId'),

    ('races', 'circuitId', 'circuits', 'circuitId'),

    ('results', 'raceId', 'races', 'raceId'),
    ('results', 'driverId', 'drivers', 'driverId'),
    ('results', 'constructorId', 'constructors', 'constructorId'),
    ('results', 'statusId', 'status', 'statusId'),


    ('sprint_results', 'raceId', 'races', 'raceId'),
    ('sprint_results', 'driverId', 'drivers', 'driverId'),
    ('sprint_results', 'constructorId', 'constructors', 'constructorId'),
    ('sprint_results', 'statusId', 'status', 'statusId'),
    
]

for table, column, parent, parent_key in foreign_keys:
    # ~ means 'not'
    missing = ~ cleaned_tables[table][column].isin(cleaned_tables[parent][parent_key])
    print(f'{table}.{column} -> {parent}: {missing.sum()} values not found')

## **Shared Drive Handling**

### **1 Driver 2 Cars**

In [ ]:
# all duplicates
temporary_results = cleaned_tables['results']
duplicates = (
    temporary_results[
        temporary_results.duplicated(
            ['raceId', 'driverId'],
            keep=False
        )
    ]
    .sort_values(['raceId', 'driverId', 'positionOrder'])
)
duplicates

In [ ]:
# we chose to keep best position order as it is the entry most likely to finish the race 
sorted_results = temporary_results.sort_values(
    ['raceId', 'driverId', 'positionOrder']
)

filtered_duplicates = sorted_results[
    sorted_results.duplicated(
        ['raceId', 'driverId'],
        keep='first'
    )
]

filtered_duplicates

In [ ]:
cleaned_tables['results'] = temporary_results.drop(
    filtered_duplicates.index
)

cleaned_tables['results']

### **1 Car 2 Drivers**

In [ ]:
# all duplicates
temporary_results = cleaned_tables['results']
duplicates = (
    temporary_results[
        temporary_results.duplicated(
            ['raceId', 'number'],
            keep=False
        )
    ]
    .sort_values(['raceId', 'number', 'positionOrder'])
)
duplicates

In [ ]:
# we chose to keep best position order as it is the entry most likely to finish the race 
sorted_results = temporary_results.sort_values(
    ['raceId', 'number', 'positionOrder']
)

filtered_duplicates = sorted_results[
    sorted_results.duplicated(
        ['raceId', 'number'],
        keep='first'
    )
]

filtered_duplicates

In [ ]:
cleaned_tables['results'] = temporary_results.drop(
    filtered_duplicates.index
)

cleaned_tables['results']

## **Post Cleaning EDA**

In [ ]:
summary = []
missing = []
sentinels = []
duplicates = []

for name, df in cleaned_tables.items():
    summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    })

    missing.append({
        "dataset": name,
        "missing_values": df.isna().sum().sum()
    })

    sentinels.append({
        "dataset": name,
        "sentinel_values": (df == r"\N").sum().sum()
    })

    duplicates.append({
        "dataset": name,
        "duplicates": df.duplicated().sum()
    })


check = cleaned_tables['results'][['raceId', 'driverId']].duplicated().any()
number = cleaned_tables['results'][['raceId', 'driverId']].duplicated().sum()
duplicate_groups = (
    cleaned_tables['results'].groupby(["raceId", "driverId"])
    .size()
)
duplicate_groups = duplicate_groups[duplicate_groups > 1]

check_car_dupe = bool(cleaned_tables['results'][['raceId', 'number']].duplicated().any())
number_car_dupe = cleaned_tables['results'][['raceId', 'number']].duplicated().sum()
duplicate_cars_groups = (
    cleaned_tables['results'].groupby(["raceId", "number"])
    .size()
)
duplicate_cars_groups = duplicate_cars_groups[duplicate_cars_groups > 1]

summary_df = pd.DataFrame(summary)
missing_df = pd.DataFrame(missing)
sentinels_df = pd.DataFrame(sentinels)
duplicates_df = pd.DataFrame(duplicates)


print('Does the results table contain any duplicate race data?: ', check)
print('How many? ', number)
print("Driver-race combinations with duplicates:", len(duplicate_groups))

print('\n')

print('Does the results table contain any duplicate car race data?: ', check_car_dupe)
print('How many? ', number_car_dupe)
print("Driver-car combinations with duplicates:", len(duplicate_cars_groups))

print('\n')

print('')
print('########### Missing Columns ####################')
for name, df in cleaned_tables.items():
    missing_cols = df.isna().sum()
    missing_cols = missing_cols[missing_cols > 0]

    if len(missing_cols) > 0:
        print(f"\n{name}")
        print(missing_cols)

print("")
print('###################### Sentinel Columns ###################')
for name, df in cleaned_tables.items():
    sentinel_cols = (df == r"\N").sum()
    sentinel_cols = sentinel_cols[sentinel_cols > 0]

    if len(sentinel_cols) > 0:
        print(f"\n{name}")
        print(sentinel_cols)


# Create a 2x2 grid of subplots with a shared figure size
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 12))

# 1. Dataset Dimensions (Top-Left)
summary_df.plot(
    x="dataset",
    y=["rows"],
    kind="bar",
    ax=axes[0, 0],
    title="Dataset Dimensions"
)

# 2. Missing Values (Top-Right)
missing_df.plot(
    x="dataset",
    y="missing_values",
    kind="bar",
    ax=axes[0, 1],
    title="Missing Values by Dataset",
    legend=False
)

# 3. Sentinel Values (Bottom-Left)
sentinels_df.plot(
    x="dataset",
    y="sentinel_values",
    kind="bar",
    ax=axes[1, 0],
    title="Sentinel Values by Dataset",
    legend=False
)

# 4. Duplicate Rows (Bottom-Right)
duplicates_df.plot(
    x="dataset",
    y="duplicates",
    kind="bar",
    ax=axes[1, 1],
    title="Duplicate Rows by Dataset",
    legend=False
)

# Rotate x-axis labels for readability and prevent overlapping titles/labels
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# for name, table in cleaned_tables.items():
#     print(f"=========== {name} ===========")
#     display(table.head(1))

display(cleaned_tables['races'].head(1))
display(cleaned_tables['circuits'].head(1))
display(cleaned_tables['results'].head(1))
display(cleaned_tables['drivers'].head(1))
display(cleaned_tables['constructor_standings'].head(1))

# **Data Engineering Questions**

## **Which circuits convert a front-row start into a podium most often, and does that ranking change after the 2014 hybrid era? Distinguish qualifying position from grid (which can differ due to penalties).**

In [ ]:
mask = (cleaned_tables['results']['grid'] <= 2) & (cleaned_tables['results']['position'] <= 3)

merged_before_2014 = (
    cleaned_tables['results'][mask]
    .merge(cleaned_tables['races'][cleaned_tables['races']['year'] < 2014][['raceId', 'circuitId']], on='raceId')
    .merge(cleaned_tables['circuits'][['circuitId', 'name']], on='circuitId')
)

merged_after_2014 = (
    cleaned_tables['results'][mask]
    .merge(cleaned_tables['races'][cleaned_tables['races']['year'] >= 2014][['raceId', 'circuitId']], on='raceId')
    .merge(cleaned_tables['circuits'][['circuitId', 'name']], on='circuitId')
)

df_before_2014 = (
    merged_before_2014.groupby('name')
    .size()
    .sort_values(ascending=False)
    .rename('num')
    .reset_index()
    .rename(columns={'name': 'circuit'})
)



df_after_2014 = (
    merged_after_2014.groupby('name')
    .size()
    .sort_values(ascending=False)
    .rename('num')
    .reset_index()
    .rename(columns={'name': 'circuit'})
)

print(f"=====================================================")
print(f"BEFORE 2014")
display(df_before_2014)
print(f"=====================================================")
print(f"=====================================================")
print(f"AFTER 2014")
display(df_after_2014)
print(f"=====================================================")

In [ ]:
top = 10
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, title, color in [
    (axes[0], df_before_2014, 'Before 2014', '#4c9be8'),
    (axes[1], df_after_2014,  '2014 onwards', '#e8744c'),
]:
    data = df.head(top).sort_values('num')   # sort so the biggest bar ends up on top
    ax.barh(data['circuit'], data['num'], color=color)
    ax.set_title(title)
    ax.set_xlabel('Front-row starts finishing on the podium')
    for i, v in enumerate(data['num']):
        ax.text(v + 0.2, i, str(v), va='center')   # value labels

fig.suptitle('Front-row starters who finished on the podium, by circuit', fontsize=14)
plt.tight_layout()
plt.show()

## **Do drivers podium more often when racing at a circuit in their home country than elsewhere, after controlling for starting position (i.e., comparing drivers who started from similar grid positions, so the effect isn’t just because home drivers happen to qualify better)?**

In [ ]:
nat_to_country = {
    'American': 'USA',
    'American-Italian': 'USA',
    'Argentine': 'Argentina',
    'Argentine-Italian': 'Argentina',
    'Argentinian ': 'Argentina',
    'Australian': 'Australia',
    'Austrian': 'Austria',
    'Belgian': 'Belgium',
    'Brazilian': 'Brazil',
    'British': 'UK',
    'Canadian': 'Canada',
    'Chilean': 'Chile',
    'Chinese': 'China',
    'Colombian': 'Colombia',
    'Czech': 'Czech Republic',
    'Danish': 'Denmark',
    'Dutch': 'Netherlands',
    'East German': 'East Germany',
    'Finnish': 'Finland',
    'French': 'France',
    'German': 'Germany',
    'Hungarian': 'Hungary',
    'Indian': 'India',
    'Indonesian': 'Indonesia',
    'Irish': 'Ireland',
    'Italian': 'Italy',
    'Japanese': 'Japan',
    'Liechtensteiner': 'Liechtenstein',
    'Malaysian': 'Malaysia',
    'Mexican': 'Mexico',
    'Monegasque': 'Monaco',
    'New Zealander': 'New Zealand',
    'Polish': 'Poland',
    'Portuguese': 'Portugal',
    'Rhodesian': 'Zimbabwe',
    'Russian': 'Russia',
    'South African': 'South Africa',
    'Spanish': 'Spain',
    'Swedish': 'Sweden',
    'Swiss': 'Switzerland',
    'Thai': 'Thailand',
    'Uruguayan': 'Uruguay',
    'Venezuelan': 'Venezuela',
}


df = (
    cleaned_tables['results'][['raceId','position','grid','driverId']]
    .merge(cleaned_tables['races'][['raceId','circuitId']] , on='raceId')
    .merge(cleaned_tables['drivers'][['driverId','nationality']], on='driverId')
    .merge(cleaned_tables['circuits'][['circuitId','country']], on='circuitId')
)

df['driver_country'] = df['nationality'].map(nat_to_country)

In [ ]:
d = df.copy()
d = d[d['grid'] >= 1]                                  # drop pit-lane starts (grid == 0)
d['home']   = d['driver_country'] == d['country']
d['podium'] = (d['position'] <= 3)                     # NaN (DNF) -> False

bins   = [0, 1, 2, 3, 5, 8, 12, 16, 40]
labels = ['P1', 'P2', 'P3', 'P4-5', 'P6-8', 'P9-12', 'P13-16', 'P17+']
d['grid_bin'] = pd.cut(d['grid'], bins=bins, labels=labels)

# podium rate per grid bin, split by home/away
g = (d.groupby(['grid_bin', 'home'], observed=True)['podium']
       .agg(rate='mean', k='sum', n='size')
       .unstack('home'))

# standard error for error bars
def se(rate, n): return np.sqrt(rate * (1 - rate) / n)

# grid-adjusted effect: actual home podium rate vs the rate home drivers
# would have had if they converted like away drivers from the same grid bin
def adjusted_effect(data):
    t = data.groupby(['grid_bin', 'home'], observed=True)['podium'].agg(['mean', 'size']).unstack('home')
    t = t.dropna()
    w = t[('size', True)]
    expected = (t[('mean', False)] * w).sum() / w.sum()
    actual   = (t[('mean', True)]  * w).sum() / w.sum()
    return actual, expected

actual, expected = adjusted_effect(d)

# bootstrap CI for (actual - expected)
rng = np.random.default_rng(0)
boots = []
for _ in range(500):
    s = d.sample(len(d), replace=True, random_state=int(rng.integers(1e9)))
    a, e = adjusted_effect(s)
    boots.append(a - e)
lo, hi = np.percentile(boots, [2.5, 97.5])

# ---- plot ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2, 1]})

x = np.arange(len(g))
wd = 0.38
away_rate, home_rate = g[('rate', False)] * 100, g[('rate', True)] * 100
away_err = se(g[('rate', False)], g[('n', False)]) * 100
home_err = se(g[('rate', True)],  g[('n', True)])  * 100

ax1.bar(x - wd/2, away_rate, wd, yerr=away_err, color='#4c9be8', label='Away', capsize=3)
ax1.bar(x + wd/2, home_rate, wd, yerr=home_err, color='#e8744c', label='Home', capsize=3)
for i, (_, r) in enumerate(g.iterrows()):
    ax1.text(i + wd/2, home_rate.iloc[i] + home_err.iloc[i] + 1, f"n={int(r[('n', True)])}",
             ha='center', fontsize=8, color='#e8744c')
ax1.set_xticks(x)
ax1.set_xticklabels(g.index)
ax1.set_xlabel('Starting grid position')
ax1.set_ylabel('Podium rate (%)')
ax1.set_title('Podium rate by grid position: home vs away')
ax1.legend()

ax2.bar(['Home drivers\nexpected\n(away rates)', 'Home drivers\nactual'],
        [expected * 100, actual * 100], color=['#9aa5b1', '#e8744c'])
ax2.set_ylabel('Podium rate (%)')
ax2.set_title(f'Grid-adjusted home effect\n{(actual-expected)*100:+.1f} pts  (95% CI {lo*100:+.1f} to {hi*100:+.1f})')

plt.tight_layout()
plt.show()

## **Which constructors have the highest mechanical-retirement rate in 2014–2021 vs. 2022–2024?**

In [ ]:
df = cleaned_tables['results'][['raceId', 'statusId','constructorId']].merge(cleaned_tables['status'], on='statusId').merge(cleaned_tables['constructors'][['constructorId', 'name']], on='constructorId').merge(cleaned_tables['races'][['raceId', 'year']], on='raceId')

mechanical_failures = ['Engine', 'Gearbox', 'Hydraulics']

all_races_by_constructor = df[df['status'].isin(mechanical_failures)]

r_2014_2021 = all_races_by_constructor[all_races_by_constructor['year'].between(2014, 2021)]
r_2022_2024 = all_races_by_constructor[all_races_by_constructor['year'].between(2022, 2024)]

# r_2014_2021.groupby('name')

In [ ]:
def failure_summary(df, start, end, min_entries=20):
    d = df[df['year'].between(start, end)]
    g = d.groupby('name').agg(
        entries=('raceId', 'size'),
        failures=('status', lambda s: s.isin(mechanical_failures).sum()),
    )
    g['rate'] = g['failures'] / g['entries'] * 100
    return g[g['entries'] >= min_entries]   # drop constructors with tiny samples

periods = [(2014, 2021, '#4c9be8'), (2022, 2024, '#e8744c')]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)

for ax, (start, end, color) in zip(axes, periods):
    s = failure_summary(df, start, end).nlargest(10, 'rate').sort_values('rate')
    ax.barh(s.index, s['rate'], color=color)
    for i, (rate, n, f) in enumerate(zip(s['rate'], s['entries'], s['failures'])):
        ax.text(rate + 0.1, i, f'{rate:.1f}% ({f}/{n})', va='center', fontsize=9)
    ax.set_title(f'{start}-{end}')
    ax.set_xlabel('Mechanical failures (% of entries)')

fig.suptitle('Constructors with the highest mechanical failure rate\n(Engine, Gearbox, Hydraulics)', fontsize=14)
plt.tight_layout()
plt.show()

# **Feature Selection & Engineering**

In [ ]:
def add_pre_race_driver_standings(modeling_data, driver_standings, races):
    """
    Add current-season pre-race points/wins and previous-season
    final points/wins for each driver.

    Current-season features use the latest standings record from
    an earlier round in the same season. If no earlier record
    exists, they default to zero.

    Previous-season features use the driver's final standings
    record from the immediately preceding season, or zero if
    unavailable.
    """

    race_order = races[['raceId', 'year', 'round']].copy()

    standings = (
        driver_standings[['raceId', 'driverId', 'points', 'wins']]
        .merge(
            race_order,
            on='raceId',
            how='left',
            validate='many_to_one'
        )
    )

    data = modeling_data.copy().reset_index(drop=True)
    data['_row_id'] = range(len(data))

    # Attach each driver's standings records from the same season.
    candidates = data[
        ['_row_id', 'driverId', 'year', 'round']
    ].merge(
        standings,
        on=['driverId', 'year'],
        how='left',
        suffixes=('', '_standing')
    )

    # Only records from earlier rounds are eligible.
    candidates = candidates[
        candidates['round_standing'] < candidates['round']
    ]

    # Latest standings record before the target race.
    current_season = (
        candidates
        .sort_values(
            ['_row_id', 'round_standing', 'raceId']
        )
        .drop_duplicates('_row_id', keep='last')
        [
            ['_row_id', 'points', 'wins']
        ]
        .rename(columns={
            'points': 'driver_points_before_race',
            'wins': 'driver_wins_before_race'
        })
    )

    data = data.merge(
        current_season,
        on='_row_id',
        how='left',
        validate='one_to_one',
        sort=False
    )

    data[
        ['driver_points_before_race', 'driver_wins_before_race']
    ] = data[
        ['driver_points_before_race', 'driver_wins_before_race']
    ].fillna(0)

    # Determine the final standings record for each driver/season.
    season_finals = (
        standings
        .sort_values(['driverId', 'year', 'round', 'raceId'])
        .drop_duplicates(['driverId', 'year'], keep='last')
        [['driverId', 'year', 'points', 'wins']]
        .rename(columns={
            'year': 'previous_season',
            'points': 'driver_previous_season_points',
            'wins': 'driver_previous_season_wins'
        })
    )

    # Match only the immediately preceding calendar season.
    data['previous_season'] = data['year'] - 1

    data = data.merge(
        season_finals,
        left_on=['driverId', 'previous_season'],
        right_on=['driverId', 'previous_season'],
        how='left',
        validate='many_to_one',
        sort=False
    )

    data[
        ['driver_previous_season_points', 'driver_previous_season_wins']
    ] = data[
        ['driver_previous_season_points', 'driver_previous_season_wins']
    ].fillna(0)

    return data.drop(columns=['_row_id', 'previous_season'])


In [ ]:
def add_driver_age(modeling_data):
    """
    Add driver age in years at the date of each target race.
    """

    data = modeling_data.copy()

    dob = pd.to_datetime(data['dob'], errors='coerce')
    race_date = pd.to_datetime(data['date'], errors='coerce')

    data['age'] = (
        (race_date - dob).dt.days / 365.2425
    )

    # Invalid dates or dates before birth are not valid ages.
    invalid_age = (data['age'] < 0) | dob.isna() | race_date.isna()
    data.loc[invalid_age, 'age'] = float('nan')
    data['age'] = data['age'].astype(int)

    return data


In [ ]:
def add_pit_stop_features(modeling_data, results, pit_stops, races):
	"""
	Add pre-race pit-stop history features.

	Fallback hierarchy:
	1. Current-season history before the target race.
	2. Previous-season driver history.
	3. Driver's historical average across all covered races.
	4. Global average across covered driver-race observations.

	A race is considered covered when pit_stops contains records for it.
	Missing driver-level stop counts within covered races are treated as zero.
	Uncovered races are excluded rather than treated as zero-stop races.
	"""

	race_info = races[['raceId', 'year', 'round']].copy()

	# Identify races for which the pit-stop table contains records.
	covered_races = pit_stops['raceId'].dropna().unique()

	# Convert stop duration to numeric seconds where possible.
	stops = pit_stops[['raceId', 'driverId', 'duration']].copy()
	stops['duration_seconds'] = pd.to_numeric(
		stops['duration'], errors='coerce'
	)

	# Summarize each driver's stops within each race.
	race_stop_stats = (
		stops.groupby(['raceId', 'driverId'], as_index=False)
		.agg(
			stops_in_race=('raceId', 'size'),
			mean_stop_duration=('duration_seconds', 'mean')
		)
	)

	# Driver race participation, restricted to covered races.
	participation = (
		results[['raceId', 'driverId']]
		.drop_duplicates()
		.merge(race_info, on='raceId', how='left', validate='many_to_one')
	)

	participation = participation[
		participation['raceId'].isin(covered_races)
	].copy()

	# Missing driver-level records in a covered race are treated as zero stops.
	participation = participation.merge(
		race_stop_stats,
		on=['raceId', 'driverId'],
		how='left',
		validate='one_to_one'
	)

	participation['stops_in_race'] = (
		participation['stops_in_race'].fillna(0)
	)

	# Calculate driver-wide historical averages.
	driver_features = (
		participation.groupby('driverId')
		.agg(
			avg_stop_duration_driver=('mean_stop_duration', 'mean'),
			avg_stops_per_race_driver=('stops_in_race', 'mean')
		)
		.reset_index()
	)

	# Calculate global averages across covered driver-race observations.
	global_avg_duration = participation['mean_stop_duration'].mean()
	global_avg_stops = participation['stops_in_race'].mean()

	# Preserve target row order.
	data = modeling_data.copy().reset_index(drop=True)
	data['_row_id'] = range(len(data))

	# Build all earlier same-season driver-race observations.
	current_candidates = data[
		['_row_id', 'driverId', 'year', 'round']
	].merge(
		participation,
		on=['driverId', 'year'],
		how='left',
		suffixes=('', '_history')
	)

	current_candidates = current_candidates[
		(current_candidates['round_history'] < current_candidates['round'])
		& current_candidates['raceId'].notna()
	]

	current_features = (
		current_candidates.groupby('_row_id')
		.agg(
			avg_stop_duration_before_race=('mean_stop_duration', 'mean'),
			avg_stops_per_race_before_race=('stops_in_race', 'mean')
		)
		.reset_index()
	)

	data = data.merge(
		current_features,
		on='_row_id',
		how='left',
		validate='one_to_one',
		sort=False
	)

	# Aggregate each driver's previous-season history.
	previous_features = (
		participation
		.groupby(['driverId', 'year'])
		.agg(
			avg_stop_duration_previous_season=('mean_stop_duration', 'mean'),
			avg_stops_per_race_previous_season=('stops_in_race', 'mean')
		)
		.reset_index()
		.rename(columns={'year': 'previous_season'})
	)

	data['previous_season'] = data['year'] - 1

	data = data.merge(
		previous_features,
		on=['driverId', 'previous_season'],
		how='left',
		validate='many_to_one',
		sort=False
	)

	# Attach driver-wide averages for fallback.
	data = data.merge(
		driver_features,
		on='driverId',
		how='left',
		validate='many_to_one',
		sort=False
	)

	# Fallback for previous-season features:
	# driver average -> global average.
	data['avg_stop_duration_previous_season'] = (
		data['avg_stop_duration_previous_season']
		.fillna(data['avg_stop_duration_driver'])
		.fillna(global_avg_duration)
	)

	data['avg_stops_per_race_previous_season'] = (
		data['avg_stops_per_race_previous_season']
		.fillna(data['avg_stops_per_race_driver'])
		.fillna(global_avg_stops)
	)

	# Fallback for current-season features:
	# previous season -> driver average -> global average.
	data['avg_stop_duration_before_race'] = (
		data['avg_stop_duration_before_race']
		.fillna(data['avg_stop_duration_previous_season'])
		.fillna(data['avg_stop_duration_driver'])
		.fillna(global_avg_duration)
	)

	data['avg_stops_per_race_before_race'] = (
		data['avg_stops_per_race_before_race']
		.fillna(data['avg_stops_per_race_previous_season'])
		.fillna(data['avg_stops_per_race_driver'])
		.fillna(global_avg_stops)
	)

	return data.drop(columns=[
		'_row_id',
		'previous_season',
		'avg_stop_duration_driver',
		'avg_stops_per_race_driver'
	])

In [ ]:
modeling_data = (
    cleaned_tables['races'][['circuitId', 'raceId', 'year', 'date', 'time', 'round']]
    .merge(cleaned_tables['circuits'][['circuitId', 'country', 'circuitRef', 'lng', 'lat', 'alt']], on = 'circuitId')
    .merge(cleaned_tables['results'][['driverId', 'constructorId', 'raceId', 'grid', 'positionOrder']], on='raceId')
    .merge(cleaned_tables['qualifying'][['raceId', 'q1']], on= 'raceId', how= 'left')
    .merge(cleaned_tables['drivers'][['driverId', 'driverRef', 'dob', 'nationality']], on='driverId')
    .merge(cleaned_tables['constructors'][['constructorId', 'constructorRef']], on= 'constructorId')
)


modeling_data.info()
modeling_data.head()

In [ ]:
q1_coverage = (
    modeling_data.groupby('year')['q1']
    .agg(total='size', available='count')
)

q1_coverage['missing_pct'] = (
    1 - q1_coverage['available'] / q1_coverage['total']
) * 100

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    q1_coverage.index,
    q1_coverage['missing_pct'],
    marker='o',
    markersize=3
)

ax.set(
    title='Missing Q1 Times by Year',
    xlabel='Year',
    ylabel='Missing Q1 values (%)',
    ylim=(0, 105)
)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## **Handling Missing Q1 Times**

In [ ]:
# 1. Calculate group-specific medians and map them to rows
group_medians = modeling_data.groupby('circuitId')['q1'].transform('median')

# 2. Fill the NaN values with the corresponding group medians
modeling_data['q1'] = modeling_data['q1'].fillna(group_medians)

#3. Fallback to global median
# global_median = modeling_data['q1'].transform('median')
modeling_data['q1'] = modeling_data['q1'].fillna('median')

In [ ]:
q1_coverage = (
    modeling_data.groupby('year')['q1']
    .agg(total='size', available='count')
)

q1_coverage['missing_pct'] = (
    1 - q1_coverage['available'] / q1_coverage['total']
) * 100

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    q1_coverage.index,
    q1_coverage['missing_pct'],
    marker='o',
    markersize=3
)

ax.set(
    title='Missing Q1 Times by Year',
    xlabel='Year',
    ylabel='Missing Q1 values (%)',
    ylim=(0, 105)
)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## **Handling Missing Race Start Times**

In [ ]:
# 1. Calculate group-specific modes and map them to rows
group_modes = modeling_data.groupby('country')['time'].transform(
	lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
)

# 2. Fill NaN values with the corresponding group mode
modeling_data['time'] = modeling_data['time'].fillna(group_modes)

# 3. Fallback to the global mode
global_mode = modeling_data['time'].mode()

if not global_mode.empty:
	modeling_data['time'] = modeling_data['time'].fillna(global_mode.iloc[0])

## **Final Feature Selection**

In [ ]:
modeling_data = add_pre_race_driver_standings(
    modeling_data=modeling_data,
    driver_standings=cleaned_tables['driver_standings'],
    races=cleaned_tables['races']
)


modeling_data = add_driver_age(modeling_data)

modeling_data = add_pit_stop_features(
    modeling_data=modeling_data,
    results=cleaned_tables['results'],
    pit_stops=cleaned_tables['pit_stops'],
    races=cleaned_tables['races']
)


modeling_data = modeling_data.drop(columns=['circuitId', 'raceId', 'driverId', 'constructorId', 'dob'])
modeling_data.info()
modeling_data.head()

# **Predictive Modeling Task**

In [ ]:
print("########## Modeling Data Pre Classification ##########")

modeling_data.info()
display(modeling_data.head())
modeling_data_backup = modeling_data.copy()

print("########## Modeling Data Post Classification ##########")

modeling_data['podium'] = modeling_data['positionOrder'].isin([1, 2, 3]).astype(int)
modeling_data.info()
display(modeling_data.head())

data = modeling_data.merge(
    races[['raceId', 'year']],
    on='raceId',
    how='left'
)

training = data[data['year'] <= 2019].copy()
validation = data[data['year'].between(2020, 2021)].copy()
testing = data[data['year'] >= 2022].copy()

training = training[modeling_data.columns]
validation = validation[modeling_data.columns]
testing = testing[modeling_data.columns]

print("Training:", training.shape)
print("Validation:", validation.shape)
print("Testing:", testing.shape)

training_percentage = (len(training)/len(modeling_data)) * 100
validation_percentage = (len(validation)/len(modeling_data)) * 100
testing_percentage = (len(testing)/len(modeling_data)) * 100

print("Training Percentage of Data:", training_percentage)
print("Validation Percentage of Data:", validation_percentage)
print("Testing Percentage of Data:", testing_percentage)

In [ ]:
numeric_features = ['grid', 'round']

categorical_features = [
    'driverId',
    'constructorId',
    'circuitId',
    'country',
    'nationality'
]
features = numeric_features + categorical_features

x_train = training[features]
y_train = training['podium']

x_val = validation[features]
y_val = validation['podium']

x_test = testing[features]
y_test = testing['podium']

display(x_train.head())
display(y_train.head())

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

x_train = preprocessor.fit_transform(x_train)
x_val = preprocessor.transform(x_val)
x_test = preprocessor.transform(x_test)

logistic_model = LogisticRegression(max_iter=1000)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

logistic_model.fit(x_train, y_train)
rf_model.fit(x_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(x_val)
print("Random Forest Test Accuracy:", accuracy_score(y_val, y_pred_rf))

print('\n')

y_pred_lm = logistic_model.predict(x_val)
print("Logistic Accuracy:", accuracy_score(y_val, y_pred_lm))
print("Logistic Precision:", precision_score(y_val, y_pred_lm))
print("Recall:", recall_score(y_val, y_pred_lm))
print("F1 Score:", f1_score(y_val, y_pred_lm))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_lm))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_lm))

cm = confusion_matrix(y_val, y_pred_lm)
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
y_prob_rf = rf_model.predict_proba(x_val)[:, 1]

roc_auc_rf = roc_auc_score(y_val, y_prob_rf)
pr_auc_rf = average_precision_score(y_val, y_prob_rf)

print("##### Random Forest #####")
print("Random Forest ROC-AUC:", roc_auc_rf)
print("Random Forest PR-AUC:", pr_auc_rf)
print("Random Forest F1 Score:", f1_score(y_val, y_pred_rf))

print("\n")
print("##### Logistic Regression #####")
y_prob_lm = logistic_model.predict_proba(x_val)[:, 1]

roc_auc_lm = roc_auc_score(y_val, y_prob_lm)
pr_auc_lm = average_precision_score(y_val, y_prob_lm)

print("Logistic Regression ROC-AUC:", roc_auc_lm)
print("Logistic Regression PR-AUC:", pr_auc_lm)
print("Logistic Regression F1 Score:", f1_score(y_val, y_pred_lm))


In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

scaler = StandardScaler()
x_train_scaled = x_train
x_val_scaled = x_val
x_test_scaled = x_test
print(f"Train: {x_train.shape}, Test: {x_test.shape}")

early_stop = EarlyStopping(monitor='val_accuracy', patience=15,
restore_best_weights=True)

def build_model():
    model = keras.Sequential([
        layers.Input(shape=(x_train_scaled.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile( optimizer='adam', loss='binary_crossentropy',
    metrics=['accuracy'] )
    return model

neural_model = build_model()
history_nm = neural_model.fit(
    x_train_scaled,
    y_train,
    validation_data=(x_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop] )

In [ ]:
plt.figure(figsize=(14, 5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_nm.history['accuracy'], label='Training Accuracy')
plt.plot(history_nm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Neural Network Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history_nm.history['loss'], label='Training Loss')
plt.plot(history_nm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Neural Network Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

y_prob_nn = neural_model.predict(x_test_scaled).ravel()
y_pred_nn = (y_prob_nn >= 0.5).astype(int)

# Metrics
accuracy = accuracy_score(y_test, y_pred_nn)
precision = precision_score(y_test, y_pred_nn, zero_division=0)
recall = recall_score(y_test, y_pred_nn, zero_division=0)
f1 = f1_score(y_test, y_pred_nn, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob_nn)
pr_auc = average_precision_score(y_test, y_prob_nn)

print("===== Neural Network Test Results =====")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_nn)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Neural Network — Test Confusion Matrix')
plt.tight_layout()
plt.show()

metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'ROC-AUC': roc_auc,
    'PR-AUC (AP)': pr_auc
}

plt.figure(figsize=(10, 5))
bars = plt.bar(metrics.keys(), metrics.values())

plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Neural Network — Test Metrics')
plt.xticks(rotation=25, ha='right')

for bar, score in zip(bars, metrics.values()):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f'{score:.3f}',
        ha='center'
    )

plt.tight_layout()
plt.show()

# **Experiments**

In [ ]:
cleaned_tables['races'].sort_values('raceId')

In [ ]:
cleaned_tables['driver_standings'][cleaned_tables['driver_standings']['driverId'] == 3].sort_values(by='raceId')

In [ ]:
temp_standings = cleaned_tables['driver_standings']
temp_standings = temp_standings[temp_standings['driverId'] == 1].merge(cleaned_tables['races'][['raceId', 'year', 'round']], on='raceId')
temp_standings = temp_standings[temp_standings['year'] >= 2023]
temp_standings.sort_values(['year', 'round'])

In [ ]:
cleaned_tables['pit_stops'].info()
print(len(cleaned_tables['pit_stops']['raceId'].unique()))
len(cleaned_tables['races'])

In [ ]:
# def add_pit_stop_features(modeling_data, results, pit_stops, races):
#     """
#     Add pre-race pit-stop history features.

#     Current-season features use eligible races before the target race.
#     Previous-season features use eligible races in the previous calendar year.

#     A race is considered covered when pit_stops contains records for it.
#     Missing driver-level stop counts within covered races are treated as zero.
#     Uncovered races are excluded rather than treated as zero-stop races.
#     """

#     race_info = races[['raceId', 'year', 'round']].copy()

#     # Identify races for which the pit-stop table contains records.
#     covered_races = pit_stops['raceId'].dropna().unique()

#     # Convert stop duration to numeric seconds where possible.
#     stops = pit_stops[['raceId', 'driverId', 'duration']].copy()
#     stops['duration_seconds'] = pd.to_numeric(
#         stops['duration'], errors='coerce'
#     )

#     # Summarize each driver's stops within each race.
#     race_stop_stats = (
#         stops.groupby(['raceId', 'driverId'], as_index=False)
#         .agg(
#             stops_in_race=('raceId', 'size'),
#             mean_stop_duration=('duration_seconds', 'mean')
#         )
#     )

#     # Driver race participation, restricted to covered races.
#     participation = (
#         results[['raceId', 'driverId']]
#         .drop_duplicates()
#         .merge(race_info, on='raceId', how='left', validate='many_to_one')
#     )

#     participation = participation[
#         participation['raceId'].isin(covered_races)
#     ].copy()

#     # Missing driver-level records in a covered race are treated as zero stops.
#     participation = participation.merge(
#         race_stop_stats,
#         on=['raceId', 'driverId'],
#         how='left',
#         validate='one_to_one'
#     )

#     participation['stops_in_race'] = (
#         participation['stops_in_race'].fillna(0)
#     )

#     # Preserve target row order.
#     data = modeling_data.copy().reset_index(drop=True)
#     data['_row_id'] = range(len(data))

#     # Build all earlier same-season driver-race observations.
#     current_candidates = data[
#         ['_row_id', 'driverId', 'year', 'round']
#     ].merge(
#         participation,
#         on=['driverId', 'year'],
#         how='left',
#         suffixes=('', '_history')
#     )

#     current_candidates = current_candidates[
#         (current_candidates['round_history'] < current_candidates['round'])
#         & current_candidates['raceId'].notna()
#     ]

#     current_features = (
#         current_candidates.groupby('_row_id')
#         .agg(
#             avg_stop_duration_before_race=('mean_stop_duration', 'mean'),
#             avg_stops_per_race_before_race=('stops_in_race', 'mean')
#         )
#         .reset_index()
#     )

#     data = data.merge(
#         current_features,
#         on='_row_id',
#         how='left',
#         validate='one_to_one',
#         sort=False
#     )

#     # Aggregate each driver's previous-season history.
#     previous_features = (
#         participation
#         .groupby(['driverId', 'year'])
#         .agg(
#             avg_stop_duration_previous_season=('mean_stop_duration', 'mean'),
#             avg_stops_per_race_previous_season=('stops_in_race', 'mean')
#         )
#         .reset_index()
#         .rename(columns={'year': 'previous_season'})
#     )

#     data['previous_season'] = data['year'] - 1

#     data = data.merge(
#         previous_features,
#         on=['driverId', 'previous_season'],
#         how='left',
#         validate='many_to_one',
#         sort=False
#     )

#     return data.drop(columns=['_row_id', 'previous_season'])

In [ ]:
# modeling_data = add_pit_stop_features(
#     modeling_data=modeling_data,
#     results=cleaned_tables['results'],
#     pit_stops=cleaned_tables['pit_stops'],
#     races=cleaned_tables['races']
# )

In [ ]:
q1_col = 'q1'  # Replace with the actual column name

modeling_data['q1_missing'] = modeling_data[q1_col].isna()

# Missingness by season
modeling_data.groupby('year')['q1_missing'].agg(
    missing_count='sum',
    total='size',
    missing_rate='mean'
).tail(20)

In [ ]:
missing_pct = (
    modeling_data.groupby('year')[q1_col]
    .apply(lambda s: s.isna().mean() * 100)
)

plt.figure(figsize=(12, 5))
plt.plot(missing_pct.index, missing_pct.values, marker='o')
plt.xlabel('Year')
plt.ylabel('Missing Q1 times (%)')
plt.title('Percentage of Missing Q1 Times by Year')
plt.xticks(sorted(modeling_data['year'].dropna().unique()), rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(modeling_data['year'].min())
print(cleaned_tables['races']['year'].min())

In [ ]:
# Work at race level to avoid counting the same race once per driver
race_times = (
    modeling_data[['raceId', 'year', 'time']]
    .drop_duplicates(subset='raceId')
    .copy()
)

# Convert HH:MM:SS strings to seconds after midnight
race_times['start_seconds'] = (
    pd.to_timedelta(race_times['time'], errors='coerce')
    .dt.total_seconds()
)

# Keep only valid clock times
valid_time = race_times['start_seconds'].between(0, 86400, inclusive='left')
race_times.loc[~valid_time, 'start_seconds'] = float('nan')

race_times['start_hour'] = race_times['start_seconds'] / 3600

# Calculate missing percentage by year
missing_by_year = (
    race_times.groupby('year')['start_seconds']
    .apply(lambda s: s.isna().mean() * 100)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of known start times
axes[0].hist(
    race_times['start_hour'].dropna(),
    bins=24,
    edgecolor='black'
)
axes[0].set(
    title='Distribution of Recorded Race Start Times',
    xlabel='Hour of day',
    ylabel='Number of races',
    xlim=(0, 24)
)
axes[0].set_xticks(range(0, 25, 2))
axes[0].grid(axis='y', alpha=0.3)

# Missing percentage by year
axes[1].plot(
    missing_by_year.index,
    missing_by_year.values,
    marker='o',
    markersize=3
)
axes[1].set(
    title='Missing Race Start Times by Year',
    xlabel='Year',
    ylabel='Missing times (%)',
    ylim=(0, 105)
)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
cleaned_tables['pit_stops'].merge(cleaned_tables['races'], on= 'raceId')['year'].min()